In [1]:
import os
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'

import findspark
findspark.init()
findspark.find()

'/opt/spark'

In [2]:
from datetime import datetime, timedelta
import sys

from pyspark import SparkContext, SparkConf
from pyspark.sql import SQLContext
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

import pyspark
from pyspark.sql import SparkSession

In [3]:
! /opt/hadoop/bin/hdfs dfs -ls /user/s18314377/analytics/user_locations/

Found 2 items
drwxr-xr-x   - s18314377 hadoop          0 2026-09-10 09:33 /user/s18314377/analytics/user_locations/date=2022-05-01
drwxr-xr-x   - s18314377 hadoop          0 2026-09-10 11:51 /user/s18314377/analytics/user_locations/date=2022-06-21


In [4]:
spark = SparkSession.builder \
                    .master("local") \
                    .appName(f"EventsPartitioningJob") \
                    .getOrCreate()

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-log4j12-1.7.30.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]


26/09/10 13:25:55 WARN Utils: Your hostname, fv4amnvfa4gfcpo9r345 resolves to a loopback address: 127.0.1.1; using 10.130.0.44 instead (on interface eth0)
26/09/10 13:25:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/10 13:25:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
df_user_locations = spark.read.parquet(
    "/user/s18314377/analytics/user_locations/date=2022-06-21/days=120/home_days=27"
)
df_user_locations.show()

In [ ]:
df_user_locations.summary().show()

In [ ]:
df_user_locations.filter(F.col("home_city").isNotNull()).show(1000)

In [ ]:
df_user_locations.show(1000)

In [ ]:
# ! /opt/hadoop/bin/hdfs dfs -mv /user/s18314377/analytics/city_statistics /user/s18314377/analytics/zone_statistics 




In [ ]:
! /opt/hadoop/bin/hdfs dfs -ls /user/s18314377/analytics/zone_statistics/

In [ ]:
df_zone_statistics = spark.read.parquet("/user/s18314377/analytics/zone_statistics/date=2022-06-21")

In [ ]:
(
    df_zone_statistics.orderBy("month", "week")
    .select("month", "week", "zone_id", "week_message", "week_reaction", "week_subscription", "week_user")
    .show(100)
)



In [ ]:
! /opt/hadoop/bin/hdfs dfs -ls -h /user/s18314377/analytics/

In [ ]:
! /opt/hadoop/bin/hdfs dfs -ls -h /user/s18314377/analytics/

In [5]:
! /opt/hadoop/bin/hdfs dfs -ls -h /user/s18314377/analytics/user_recommendations

Found 2 items
drwxr-xr-x   - s18314377 hadoop          0 2026-09-10 10:59 /user/s18314377/analytics/user_recommendations/date=2022-05-01
drwxr-xr-x   - s18314377 hadoop          0 2026-09-10 12:59 /user/s18314377/analytics/user_recommendations/date=2022-06-21


In [6]:
df_user_recommendations = spark.read.parquet("/user/s18314377/analytics/user_recommendations/date=2022-06-21")

In [7]:
df_user_recommendations.summary().show()

+-------+-----------------+------------------+----------+-----+-------------------+
|summary|        user_left|        user_right|   zone_id| days|maximum_distance_km|
+-------+-----------------+------------------+----------+-----+-------------------+
|  count|            32682|             32682|     32682|32682|              32682|
|   mean|54913.97738816474|110990.50483446545|      null|120.0|                1.0|
| stddev|39346.69884875962| 39592.20192583921|      null|  0.0|                0.0|
|    min|                1|            100001|  Adelaide|  120|                1.0|
|    25%|          21468.0|           82787.0|      null|  120|                1.0|
|    50%|          48013.0|          117775.0|      null|  120|                1.0|
|    75%|          82381.0|          144151.0|      null|  120|                1.0|
|    max|            99999|             99999|Wollongong|  120|                1.0|
+-------+-----------------+------------------+----------+-----+-------------

In [8]:
df_user_recommendations.show(1000)

+---------+----------+-------------------+-----------+-------------------+----+-------------------+
|user_left|user_right|     processed_dttm|    zone_id|         local_time|days|maximum_distance_km|
+---------+----------+-------------------+-----------+-------------------+----+-------------------+
|   101074|    119258|2022-06-21 23:59:59|   Brisbane|2021-03-15 03:12:58| 120|                1.0|
|   101515|    150019|2022-06-21 23:59:59|  Melbourne|2021-03-26 17:50:05| 120|                1.0|
|   101908|    151587|2022-06-21 23:59:59| Launceston|2021-04-13 06:35:27| 120|                1.0|
|   102636|    106668|2022-06-21 23:59:59|   Brisbane|2021-02-23 11:49:50| 120|                1.0|
|    10309|     90190|2022-06-21 23:59:59|   Maitland|2022-05-29 01:32:53| 120|                1.0|
|   103258|    137434|2022-06-21 23:59:59|      Perth|2021-04-23 11:30:34| 120|                1.0|
|   103822|    142509|2022-06-21 23:59:59|   Brisbane|2021-03-25 16:19:35| 120|                1.0|


In [ ]:
! /opt/hadoop/bin/hdfs dfs -ls -h /user/master/data/geo/events

In [ ]:
from datetime import date

In [ ]:
date(2022, 6, 21) - date(2022, 1 , 1)